# ESIM, step 1: word vectors

Trains the word-vector tables ESIM reads, on the address text of all three splits (both sides, no labels): word2vec (skip-gram, 200-d, `window=2`, 50 epochs) with `min_count` 2 and 1, and fastText with `min_count` 1 (same settings + character 3–6-grams). Saved to `checkpoints/esim/` (git-ignored); `02_train.ipynb` loads them.

§2 also shows the validation AUC behind the epoch choice (5 / 20 / 50). §4 is the cosine check quoted in the paper's ESIM section (same-street rare house numbers vs random number pairs).

Note: gensim training is only reproducible with one worker *and* a fixed `PYTHONHASHSEED` set before the kernel starts; the runs in the paper did not fix it, so a rebuilt table differs slightly from the original.

## 0. Setup
**Why:** one place for imports and the dataset name every cell uses.

In [ ]:
# Setup: imports and dataset name
%load_ext autoreload
%autoreload 2

import re
import sys
from collections import Counter
sys.path.append("..")   # repository root -> lets us import helpers/ and the model packages

import numpy as np
import pandas as pd
from gensim.models import Word2Vec, FastText
from sklearn.metrics import roc_auc_score

from helpers.pairs import load_pairs
import os
from esim.model import tokenize, CHECKPOINTS   # same tokenisation ESIM uses

pd.set_option("display.width", 250)
pd.set_option("display.expand_frame_repr", False)

DATASET = "overture_case2s"
os.makedirs(CHECKPOINTS, exist_ok=True)

## 1. Load the pairs

In [ ]:
# Load: train / val / test pairs
train, train_label = load_pairs(DATASET, "train")
val, val_label = load_pairs(DATASET, "val")
test, test_label = load_pairs(DATASET, "test")
full_dataset = pd.concat([train, val, test])
pd.concat([train, val, test]).groupby(["split", "neg_type"], observed = True).size().unstack()

### Generating Corpus
- Combine all addresses from `address_a` and `address_b`
- Remove `address` dupes, because they can appear multiple times dataset. For example, can exist in pos and neg
- Dedupe on id, to just avoid row duplication

In [ ]:
id_and_address = pd.concat([
    full_dataset[["address_a", "id_a"]].rename(columns = {
        "address_a":"address",
        "id_a": "id",
    }),
    full_dataset[["address_b", "id_b"]].rename(columns = {
        "address_b": "address",
        "id_b": "id"
    })
]).drop_duplicates(subset = ["id", "address"])["address"]

### 2. word2vec: choose the epoch count
- **Tokenize**: returns a sequence of words/numbers and then upper-cases them

- **Address_vector**: iterates over all tokens using model embeddings. 
    - all unknown embeddings get ignored
    - keep model safe by never inputting an empty list i.e. `if not known`
    - Then: actually run embedding, and squash all word embeddings in sentence into one

- **Pair Cosines**: 
    - Runs a dot product between `address_a` and `address_b`. If the model is correct, we should expect their similarity to be high

- **AUC**
    - We generate a `pair_cosines`dataset with appropriate labels
    - Running `roc_auc` will tests and give a threshold distribution, of how different thresholds do in correctly matching all neg/pos pairs

In [ ]:
# word2vec helpers: tokenize, address vector, pair cosines, AUC
# 1 tokenize: esim/model.py (imported in Setup), the same one ESIM uses
# 2
def address_vector(model, tokens):
    known = [t for t in tokens if t in model.wv.key_to_index]
    if not known:
        return np.zeros(model.vector_size)
    return model.wv[known].mean(axis = 0)
# 3
def pair_cosines(model, df):
    va = np.stack([address_vector(model, tokenize(s)) for s in df["address_a"]])
    vb = np.stack([address_vector(model, tokenize(s)) for s in df["address_b"]])
    return (va * vb).sum(axis = 1) / (np.linalg.norm(va, axis =1) * np.linalg.norm(vb, axis =1) + 1e-9)

def auc(model, pos, neg):
    labels = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))])
    scores = np.concatenate([
        pair_cosines(model, pos),
        pair_cosines(model, neg)
    ])
    return roc_auc_score(labels, scores)

### 2b. Epoch loop: val AUC (non-identical matches vs case 2s / vs case 3) at 5 / 20 / 50 epochs; the 50-epoch models are saved

In [ ]:
# Epoch choice: train at 5 / 20 / 50 epochs, val AUC per setting

sentences = id_and_address.map(tokenize).tolist()
results = {}

val_case2s = val[val["neg_type"] == "case2s"]
val_case3 = val[val["neg_type"] == "case3"]
val_pos = val[(val["neg_type"] == "case0") & (val["address_a"] != val["address_b"])]

for min_count in [1, 2]:
    for epochs in [5, 20, 50]:
        model = Word2Vec(sentences, vector_size=200, sg=1, window = 2, min_count=min_count, epochs= epochs, seed=0, workers=1)
        
        results[epochs] = {"case2s": auc(model, val_pos, val_case2s), "case3": auc(model, val_pos, val_case3)}
    assert model.epochs == 50
    model.save(os.path.join(CHECKPOINTS, f"word2vec_50_mc{min_count}.model"))
    display(pd.DataFrame(results).T)

### 3. fastText variant
Same settings as word2vec `min_count=1` (skip-gram, 200-d, `window=2`, 50 epochs), plus character n-grams of length 3–6 (gensim defaults). A word's vector = its own vector + the vectors of its n-grams, so `4817` and `4819` share part of their vector.

In [ ]:
# fastText: same settings as word2vec min_count=1, plus char n-grams 3-6
# Self-contained (needs Setup, Load, corpus, helpers), so the 6-model epoch loop above need not be rerun
sentences = id_and_address.map(tokenize).tolist()
val_case2s = val[val["neg_type"] == "case2s"]
val_case3 = val[val["neg_type"] == "case3"]
val_pos = val[(val["neg_type"] == "case0") & (val["address_a"] != val["address_b"])]

ft = FastText(sentences, vector_size=200, sg=1, window=2, min_count=1, epochs=50, min_n=3, max_n=6, seed=0, workers=1)
print({"case2s": auc(ft, val_pos, val_case2s), "case3": auc(ft, val_pos, val_case3)})   # same val AUCs as the word2vec table
ft.save(os.path.join(CHECKPOINTS, "fasttext_50_mc1.model"))

## 4. Cosine check: same-street rare house numbers vs random numbers (paper, ESIM section)
Val case 1 pairs whose two house numbers are both outside the `min_count` 2 vocabulary and whose rest of the address is identical (161 pairs), against 1,000 random pairs of house numbers. Mean cosine similarity of the two numbers' vectors, for word2vec `min_count` 1 and for fastText.

In [ ]:
import random
# min_count=1 collision check: case1 rare-number pairs vs random house-number pairs
old_vocab = Word2Vec.load(os.path.join(CHECKPOINTS, "word2vec_50_mc2.model")).wv.key_to_index   # the vocab ESIM was trained with
new_wv = Word2Vec.load(os.path.join(CHECKPOINTS, "word2vec_50_mc1.model")).wv

# Group: val case1 rows, both first tokens are numbers unknown to the old vocab, rest of the tokens identical
group_pairs = []
for a, b in zip(val.loc[val["neg_type"] == "case1", "address_a"], val.loc[val["neg_type"] == "case1", "address_b"]):
    ta, tb = tokenize(a), tokenize(b)
    if (ta and tb and ta[0].isdigit() and tb[0].isdigit()
            and ta[0] not in old_vocab and tb[0] not in old_vocab and ta[1:] == tb[1:]
            and ta[0] != tb[0]):   # two different numbers: never a word compared with itself
        group_pairs.append((ta[0], tb[0]))
print(len(group_pairs), "rare-number case 1 pairs in val")   # 161 on the paper's data

# Baseline: random pairs of house numbers (digits only, no leading 0 -> drops ZIP-like tokens)
house_numbers = [k for k in new_wv.key_to_index if k.isdigit() and not k.startswith("0")]
rng = random.Random(0)
random_pairs = [tuple(rng.sample(house_numbers, 2)) for _ in range(1000)]

sims = pd.DataFrame({
    "case1 rare-number group": pd.Series([new_wv.similarity(a, b) for a, b in group_pairs]),
    "random numbers (n=1000)": pd.Series([new_wv.similarity(a, b) for a, b in random_pairs]),
})
display(sims.describe().T)
print("share with similarity > 0.9:")
print(sims.apply(lambda col: (col.dropna() > 0.9).mean()))   # dropna: the group column is shorter (NaN-padded)

In [ ]:
# fastText collision check: reuses group_pairs / random_pairs from the cell above
ft_wv = FastText.load(os.path.join(CHECKPOINTS, "fasttext_50_mc1.model")).wv
ft_sims = pd.DataFrame({
    "case1 rare-number group": pd.Series([ft_wv.similarity(a, b) for a, b in group_pairs]),
    "random numbers (n=1000)": pd.Series([ft_wv.similarity(a, b) for a, b in random_pairs]),
})
display(ft_sims.describe().T)
print("share with similarity > 0.9:")
print(ft_sims.apply(lambda col: (col.dropna() > 0.9).mean()))